# 03 · Explore SEC EDGAR financials (Decision 2b)

**Goal:** decide whether SEC EDGAR's free `companyfacts` API is enough for the fundamentals side of Spin Check, using **Macy's** as the test company (we already have its transcripts).

**Questions this notebook answers:**
1. Which revenue tag(s) does Macy's use, and does it switch around 2018 (ASC 606)?
2. For **fiscal Q1 2018** (quarter ended 2018-05-05), how many entries exist, and do any values differ (restatements)?
3. Is there a gross profit tag, or must it be revenue − cost of goods sold?
4. What is the inventory tag?
5. Can Q4 be derived as annual − (Q1 + Q2 + Q3)?

**Rules:** every response is cached in `data/raw/sec/`; the SEC requires a `User-Agent` with your name and email (stored in `.env`, never in this notebook); stay well under 10 requests/second.

In [1]:
import json
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv

ENV_PATH = find_dotenv(usecwd=True)  # same pattern as notebook 02
assert ENV_PATH, ".env not found - create it in the project root"
load_dotenv(ENV_PATH)
PROJECT_ROOT = Path(ENV_PATH).parent

# The SEC blocks requests without a descriptive User-Agent: "Your Name your@email.com".
# Add this line to .env:  SEC_USER_AGENT="Your Name your@email.com"
USER_AGENT = os.getenv("SEC_USER_AGENT")
assert USER_AGENT and "@" in USER_AGENT, 'add SEC_USER_AGENT="Your Name your@email.com" to .env'
print("User-Agent set (not printed - it contains your email)")

User-Agent set (not printed - it contains your email)


/Users/vrushankhitenrawal/Desktop/spin-detector/env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "sec"
RAW_DIR.mkdir(parents=True, exist_ok=True)
HEADERS = {"User-Agent": USER_AGENT}


def sec_get_json(url, cache_name):
    # cache-first, like fetch_transcript() in notebook 02: a URL is downloaded once, ever
    path = RAW_DIR / cache_name
    if path.exists():
        return json.loads(path.read_text())

    response = requests.get(url, headers=HEADERS, timeout=30)
    time.sleep(0.2)  # max ~5 requests/second, half the SEC's limit
    if not response.ok:
        raise RuntimeError(f"{url} -> HTTP {response.status_code}")  # no key in this URL, safe to show
    data = response.json()
    path.write_text(json.dumps(data))
    return data

### Step 1: find Macy's CIK
The SEC publishes `company_tickers.json`: a ticker → CIK map. **Gotcha:** it lists only *current* tickers, so delisted companies (e.g. Bed Bath & Beyond) may be missing. The second check below shows whether that's true.

In [3]:
tickers = sec_get_json("https://www.sec.gov/files/company_tickers.json", "company_tickers.json")
ticker_df = pd.DataFrame(tickers.values())  # the JSON is {"0": {...}, "1": {...}}, so take the values

print(ticker_df[ticker_df["ticker"] == "M"])
print()
print("BBBY present in the current-ticker file?", "BBBY" in set(ticker_df["ticker"]))
print(ticker_df[ticker_df["ticker"] == "BBBY"])  # if a row shows up, check the title: is it really Bed Bath & Beyond?

      cik_str ticker         title
1234   794367      M  Macy's, Inc.

BBBY present in the current-ticker file? False
Empty DataFrame
Columns: [cik_str, ticker, title]
Index: []


In [4]:
macy_row = ticker_df[ticker_df["ticker"] == "M"].iloc[0]
CIK = int(macy_row["cik_str"])
CIK10 = f"{CIK:010d}"  # the API wants the CIK zero-padded to 10 digits
print(macy_row["title"], "-> CIK", CIK10)

Macy's, Inc. -> CIK 0000794367


### Step 2: fetch Macy's full XBRL history (1 request)

In [5]:
facts = sec_get_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK10}.json", f"companyfacts_{CIK10}.json")

print("entity:", facts["entityName"])
print("taxonomies:", list(facts["facts"].keys()))
us_gaap = facts["facts"]["us-gaap"]
print("us-gaap tags:", len(us_gaap))

entity: Macy's, Inc.
taxonomies: ['dei', 'us-gaap', 'ffd']
us-gaap tags: 539


### Step 3: which tags exist for revenue, cost, gross profit and inventory?
Search tag names instead of guessing them. For each candidate: how many USD entries, and which period-end years it covers.

In [6]:
def tag_overview(keywords):
    rows = []
    for tag, body in us_gaap.items():
        if not any(k.lower() in tag.lower() for k in keywords):
            continue
        entries = body.get("units", {}).get("USD", [])
        if not entries:
            continue  # skip tags reported in other units (shares, %)
        ends = sorted(e["end"] for e in entries)
        rows.append({"tag": tag, "entries": len(entries), "first_end": ends[0], "last_end": ends[-1]})
    return pd.DataFrame(rows).sort_values("entries", ascending=False) if rows else pd.DataFrame()


pd.set_option("display.max_colwidth", 80)
for group, keywords in {
    "REVENUE": ["Revenue", "SalesRevenue"],
    "COST OF SALES": ["CostOfGoods", "CostOfRevenue", "CostOfMerchandise"],
    "GROSS PROFIT": ["GrossProfit"],
    "INVENTORY": ["Inventory"],
}.items():
    print(f"===== {group} =====")
    print(tag_overview(keywords).head(10).to_string(index=False))
    print()

===== REVENUE =====
                                                                                                tag  entries  first_end   last_end
                                                                                    SalesRevenueNet      185 2008-02-02 2018-08-04
                                                RevenueFromContractWithCustomerExcludingAssessedTax      128 2017-01-28 2026-08-01
                                                                                           Revenues       45 2022-01-29 2026-08-01
                                                     ContractWithCustomerLiabilityRevenueRecognized       24 2017-01-28 2026-01-31
                                                      OperatingLeasesIncomeStatementSubleaseRevenue       18 2013-02-02 2019-02-02
ContractWithCustomerLiabilityCumulativeCatchUpAdjustmentToRevenueChangeInEstimateOfTransactionPrice       11 2019-02-02 2025-02-01

===== COST OF SALES =====
                       tag  entries 

### Step 4: turn one tag into a table
Each entry has: `start` (duration facts only), `end`, `val`, `filed`, `form`, `fy`, `fp`, `accn`.
- `duration_days` tells a **quarter (~91 days)** from **year-to-date (~182/273)** from **annual (~364)**.
- **Gotcha:** `fy`/`fp` describe the *filing*, not the number. A Q1 2019 10-Q also carries Q1 2018 comparison figures labelled `fy: 2019`. Use `start`/`end` to identify the period.

In [7]:
def facts_df(tag):
    entries = us_gaap.get(tag, {}).get("units", {}).get("USD", [])
    df = pd.DataFrame(entries)
    if df.empty:
        return df
    df["tag"] = tag
    df["end"] = pd.to_datetime(df["end"])
    df["filed"] = pd.to_datetime(df["filed"])
    if "start" in df:
        df["start"] = pd.to_datetime(df["start"])
        df["duration_days"] = (df["end"] - df["start"]).dt.days
    return df.sort_values(["end", "filed"])


# fill in from the step-3 output: Macy's revenue tags, in the order you'd try them
REVENUE_TAGS = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",  # post-2018 (ASC 606) standard tag
    "SalesRevenueNet",                                     # common pre-2018 tag
    "Revenues",
    "SalesRevenueGoodsNet",
]
revenue = pd.concat([facts_df(t) for t in REVENUE_TAGS if t in us_gaap], ignore_index=True)

# which tag covers which years? (answers question 1: is there a switch around 2018?)
quarters = revenue[revenue["duration_days"].between(80, 100)]
print(quarters.groupby("tag")["end"].agg(["min", "max", "count"]))

                                                           min        max  \
tag                                                                         
RevenueFromContractWithCustomerExcludingAssessedTax 2017-04-29 2026-08-01   
Revenues                                            2022-04-30 2026-08-01   
SalesRevenueNet                                     2008-08-02 2018-08-04   

                                                     count  
tag                                                         
RevenueFromContractWithCustomerExcludingAssessedTax     72  
Revenues                                                22  
SalesRevenueNet                                        120  


### Step 5: fiscal Q1 2018 deep dive (restatements?)
Macy's fiscal Q1 2018 ended **2018-05-05**. Every entry for that quarter, oldest filing first. The **first-reported** value is the earliest `filed` row: that's what was public at the time, and the only value the model may use.

In [8]:
q1_2018 = quarters[(quarters["end"] >= "2018-04-28") & (quarters["end"] <= "2018-05-12")]
print(q1_2018[["tag", "start", "end", "val", "filed", "form", "fy", "fp", "accn"]].to_string(index=False))

print()
if q1_2018.empty:
    print("no quarter-length revenue entries ending near 2018-05-05 - check REVENUE_TAGS against step 3")
else:
    print("distinct values reported for this quarter:", q1_2018["val"].nunique())
    first = q1_2018.sort_values("filed").iloc[0]
    print(f"first-reported: {first['val']:,} (filed {first['filed'].date()} in {first['form']})")

                                                tag      start        end        val      filed form   fy fp                 accn
RevenueFromContractWithCustomerExcludingAssessedTax 2018-02-04 2018-05-05 5541000000 2019-04-03 10-K 2018 FY 0000794367-19-000038
RevenueFromContractWithCustomerExcludingAssessedTax 2018-02-04 2018-05-05 5541000000 2019-06-05 10-Q 2019 Q1 0000794367-19-000080
RevenueFromContractWithCustomerExcludingAssessedTax 2018-02-04 2018-05-05 5541000000 2020-03-30 10-K 2019 FY 0000794367-20-000040
                                    SalesRevenueNet 2018-02-04 2018-05-05 5541000000 2018-06-01 10-Q 2018 Q1 0000794367-18-000078

distinct values reported for this quarter: 1
first-reported: 5,541,000,000 (filed 2018-06-01 in 10-Q)


### Step 6: inventory (an *instant* fact: no `start`, just a balance on the `end` date)

In [9]:
INVENTORY_TAG = "InventoryNet"  # check against the step-3 output; change if Macy's uses another tag
inventory = facts_df(INVENTORY_TAG)
if inventory.empty:
    print(f"{INVENTORY_TAG} not found - pick the right tag from step 3")
else:
    print(inventory[(inventory["end"] >= "2018-04-28") & (inventory["end"] <= "2018-05-12")]
          [["end", "val", "filed", "form", "fy", "fp"]].to_string(index=False))

       end        val      filed form   fy fp
2018-05-05 5291000000 2018-06-01 10-Q 2018 Q1
2018-05-05 5291000000 2019-06-05 10-Q 2019 Q1


### Step 7: gross profit: tagged, or revenue − cost of sales?

In [10]:
print("GrossProfit tag present:", "GrossProfit" in us_gaap)

COST_TAGS = ["CostOfGoodsAndServicesSold", "CostOfRevenue", "CostOfGoodsSold"]  # adjust from step 3
cost = pd.concat([facts_df(t) for t in COST_TAGS if t in us_gaap], ignore_index=True)
if cost.empty:
    print("no cost-of-sales tag from COST_TAGS - check step 3 (retailers sometimes use custom tags)")
else:
    cost_q = cost[cost["duration_days"].between(80, 100)]
    print(cost_q.groupby("tag")["end"].agg(["min", "max", "count"]))

GrossProfit tag present: True
                                  min        max  count
tag                                                    
CostOfGoodsAndServicesSold 2008-08-02 2026-08-01    195


### Step 8: Q4 is never a quarter. Derive it.
The 10-K reports the **full fiscal year**, so Q4 = annual − (Q1 + Q2 + Q3), using **first-reported** values only. Macy's fiscal 2018 ran roughly Feb 4, 2018 → Feb 2, 2019.

In [11]:
def first_reported(df):
    # one row per period: the earliest filing for each (start, end)
    return df.sort_values("filed").groupby(["start", "end"], as_index=False).first()


fy_start, fy_end = pd.Timestamp("2018-01-28"), pd.Timestamp("2019-02-09")  # window around fiscal 2018
in_fy = first_reported(revenue[(revenue["start"] >= fy_start) & (revenue["end"] <= fy_end)])

annual = in_fy[in_fy["duration_days"].between(350, 380)]
q123 = in_fy[in_fy["duration_days"].between(80, 100)].sort_values("end")
print("annual:\n", annual[["start", "end", "val", "filed"]].to_string(index=False))
print("quarters found:\n", q123[["start", "end", "val", "filed"]].to_string(index=False))

if len(annual) == 1 and len(q123) == 3:
    q4 = annual["val"].iloc[0] - q123["val"].sum()
    print(f"\nderived Q4 revenue: {q4:,}  (public on {annual['filed'].iloc[0].date()}, the 10-K filing date)")
else:
    print("\nnot exactly 1 annual + 3 quarters - inspect the rows above before deriving Q4")

annual:
      start        end         val      filed
2018-02-04 2019-02-02 24971000000 2019-04-03
quarters found:
      start        end        val      filed
2018-02-04 2018-05-05 5541000000 2018-06-01
2018-05-06 2018-08-04 5572000000 2018-08-31
2018-08-05 2018-11-03 5404000000 2018-12-07
2018-11-04 2019-02-02 8455000000 2019-04-03

not exactly 1 annual + 3 quarters - inspect the rows above before deriving Q4


### Record the results (filled 2026-10-20)
| Question | Answer | Evidence |
|---|---|---|
| 1. Revenue tag(s), switch around 2018? | **Yes.** `SalesRevenueNet` until 2018-08-04 → `RevenueFromContractWithCustomerExcludingAssessedTax` from fiscal 2018 (ASC 606), with prior-year comparisons back to 2017-04. `Revenues` also appears from 2022, but it's **net sales + other revenue**: consistently ~$160–200M/quarter higher (e.g. 2026-08-01: 5,059M vs 4,866M) → **never mix the two** | step 3, step 4 (+ side check) |
| 2. Fiscal Q1 2018: entries / distinct values / first-reported | **4 entries, 1 distinct value** ($5,541M), no restatement. **First-reported: 10-Q filed 2018-06-01 under the OLD tag** `SalesRevenueNet`; later filings repeat it under the new tag (earliest 2019-04-03) → first-reported must be computed **across tags**, or the number looks public 10 months late | step 5 |
| 3. Gross profit tagged, or revenue − cost? | `GrossProfit` exists but **stops at 2018-02-03**. `CostOfGoodsAndServicesSold` runs continuously 2008 → 2026 → **always compute gross profit = net sales − COGS** (one definition for every year) | step 3, step 7 |
| 4. Inventory tag | `InventoryFinishedGoods` (2008 → 2018-02) then `InventoryNet` (2016-07 → now) → **tag list, newest first**. Q1 2018 (2018-05-05): **$5,291M**, same value in both filings | step 3, step 6 |
| 5. Q4 derivable? | **Q4 is tagged** in the 10-K: $8,455M (2018-11-04 → 2019-02-02, filed 2019-04-03). Derived: 24,971 − (5,541 + 5,572 + 5,404) = **$8,454M**, within $1M (rounding). → **use tagged Q4; derive only as fallback; cross-check when both exist.** (My "Q4 is never a quarter" assumption was wrong for Macy's; the cell's "not exactly 3 quarters" message is because it found 4.) | step 8 |
| BBBY in the current-ticker file? | **No** → `company_tickers.json` covers current tickers only; delisted companies need a CIK lookup by name on EDGAR | step 1 |

**Requests used:** 2 (ticker file + Macy's companyfacts), both cached in `data/raw/sec/`.

**Conclusion:** SEC companyfacts is enough for the fundamentals: free, official, full history in one request per company, and every value carries a `filed` date for point-in-time control. It needs careful handling, not a different source.

**Design consequences for Decision 2b:**
1. Per concept, an **ordered tag list** that keeps **one definition**: net sales = `RevenueFromContractWithCustomerExcludingAssessedTax` → `SalesRevenueNet` → (`Revenues` only if neither exists, flagged).
2. **First-reported across all tags of a concept**: earliest `filed` per (`start`, `end`).
3. **Gross margin computed**, never read from `GrossProfit`.
4. **Q4**: tagged value first, derived as fallback, with a cross-check.
5. **Identify periods by `start`/`end`**, never `fy`/`fp`; quarters = 80–100 days.
6. **Timing (open, Decision 4):** the 10-K was filed 2019-04-03, ~5 weeks after the Q4 earnings call. XBRL `filed` lags the call → decide whether the call/press-release date or the filing date counts as "public". Safe default: filing date.